In [ ]:

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import re
from collections import Counter
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

file_path = "Womens Clothing E-Commerce Reviews.csv"

df = pd.read_csv(file_path)

print("=" * 60)
print("LSTM CUSTOMER REVIEW SENTIMENT ANALYZER")
print("=" * 60)

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())
review_column = "Review Text"
rating_column = "Rating"

df = df.dropna(
    subset=[review_column, rating_column]
).copy()

def create_sentiment(rating):

    if rating <= 2:
        return 0

    elif rating >= 4:
        return 1

    else:
        return -1


df["sentiment"] = df[
    rating_column
].apply(create_sentiment)


# Remove neutral reviews

df = df[
    df["sentiment"] != -1
].copy()

df = df[
    [review_column, "sentiment"]
].copy()

df.columns = [
    "review",
    "sentiment"
]

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["review"] = df[
    "review"
].apply(clean_text)

df = df[
    df["review"].str.len() > 0
].reset_index(drop=True)


print("\nCleaned Dataset Shape:")
print(df.shape)

print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["review"].values,
    df["sentiment"].values,
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"].values
)

MAX_VOCAB_SIZE = 20000

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

counter = Counter()

for sentence in X_train:

    words = sentence.split()

    counter.update(words)


most_common_words = counter.most_common(
    MAX_VOCAB_SIZE - 2
)


word_to_index = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for index, (word, count) in enumerate(
    most_common_words,
    start=2
):

    word_to_index[word] = index

vocab_size = len(
    word_to_index
)

print("\nVocabulary Size:")
print(vocab_size)

MAX_LENGTH = 100
def text_to_sequence(text):
    words = text.split()
    sequence = []
    for word in words:
        sequence.append(
            word_to_index.get(
                word,
                word_to_index[UNK_TOKEN]
            )
        )

    sequence = sequence[
        :MAX_LENGTH
    ]
    while len(sequence) < MAX_LENGTH:

        sequence.append(
            word_to_index[PAD_TOKEN]
        )
    return sequence
X_train_sequences = np.array(
    [
        text_to_sequence(text)
        for text in X_train
    ],
    dtype=np.int64
)
X_test_sequences = np.array(
    [
        text_to_sequence(text)
        for text in X_test
    ],
    dtype=np.int64
)
y_train = np.array(
    y_train,
    dtype=np.float32
)
y_test = np.array(
    y_test,
    dtype=np.float32
)
print("\nTraining Shape:")
print(X_train_sequences.shape)

print("\nTesting Shape:")
print(X_test_sequences.shape)
class ReviewDataset(Dataset):

    def __init__(
        self,
        sequences,
        labels
    ):
        self.sequences = torch.tensor(
            sequences,
            dtype=torch.long
        )
        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )
    def __len__(self):

        return len(
            self.labels
        )
    def __getitem__(self, index):

        return (
            self.sequences[index],
            self.labels[index]
        )
train_dataset = ReviewDataset(
    X_train_sequences,
    y_train
)
test_dataset = ReviewDataset(
    X_test_sequences,
    y_test
)
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
class LSTMSentimentModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3
    ):

        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        self.dropout = nn.Dropout(
            dropout
        )
        self.fc = nn.Linear(
            hidden_dim * 2,
            1
        )
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = (
            self.lstm(embedded)
        )
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        hidden_concat = torch.cat(
            (
                forward_hidden,
                backward_hidden
            ),
            dim=1
        )
        hidden_concat = self.dropout(
            hidden_concat
        )
        output = self.fc(
            hidden_concat
        )
        return output.squeeze(1)
model = LSTMSentimentModel(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3
)
model = model.to(device)
print("\n" + "=" * 60)
print("LSTM MODEL")
print("=" * 60)

print(model)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
def train_model(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for sequences, labels in loader:
        sequences = sequences.to(
            device
        )
        labels = labels.to(
            device
        )
        optimizer.zero_grad()
        outputs = model(
            sequences
        )
        loss = criterion(
            outputs,
            labels
        )
        loss.backward()
        optimizer.step()
        total_loss += (
            loss.item()
            *
            sequences.size(0)
        )
        predictions = (
            torch.sigmoid(outputs)
            >= 0.5
        ).float()
        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)
    average_loss = (
        total_loss / total
    )
    accuracy = (
        correct / total
    )
    return average_loss, accuracy
def evaluate_model(
    model,
    loader,
    criterion
):

    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for sequences, labels in loader:
            sequences = sequences.to(
                device
            )

            labels = labels.to(
                device
            )
            outputs = model(
                sequences
            )
            loss = criterion(
                outputs,
                labels
            )

            total_loss += (
                loss.item()
                *
                sequences.size(0)
            )
            probabilities = torch.sigmoid(
                outputs
            )


            predictions = (
                probabilities >= 0.5
            ).long()


            all_predictions.extend(
                predictions.cpu().numpy()
            )


            all_labels.extend(
                labels.cpu().numpy()
            )


    average_loss = (
        total_loss
        /
        len(loader.dataset)
    )


    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )


    return (
        average_loss,
        accuracy,
        all_labels,
        all_predictions
    )

EPOCHS = 5
print("\n" + "=" * 60)
print("TRAINING LSTM")
print("=" * 60)
for epoch in range(
    EPOCHS
):
    train_loss, train_accuracy = (
        train_model(
            model,
            train_loader,
            criterion,
            optimizer
        )
    )
    test_loss, test_accuracy, _, _ = (
        evaluate_model(
            model,
            test_loader,
            criterion
        )
    )
    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Accuracy: {train_accuracy:.4f} "
        f"Test Loss: {test_loss:.4f} "
        f"Test Accuracy: {test_accuracy:.4f}"
    )
test_loss, test_accuracy, y_true, y_pred = (
    evaluate_model(
        model,
        test_loader,
        criterion
    )
)

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(
    "Test Loss:",
    round(test_loss, 4)
)

print(
    "Test Accuracy:",
    round(test_accuracy * 100, 2),
    "%"
)
print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Negative",
            "Positive"
        ]
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)
def predict_sentiment(
    review
):

    cleaned_review = clean_text(
        review
    )
    sequence = text_to_sequence(
        cleaned_review
    )
    sequence_tensor = torch.tensor(
        [sequence],
        dtype=torch.long
    ).to(device)
    model.eval()
    with torch.no_grad():

        output = model(
            sequence_tensor
        )


        probability = torch.sigmoid(
            output
        ).item()


    if probability >= 0.5:

        sentiment = "Positive"

    else:

        sentiment = "Negative"


    return sentiment, probability


# ============================================================
# 26. TEST REVIEWS
# ============================================================

print("\n" + "=" * 60)
print("TEST REVIEW PREDICTIONS")
print("=" * 60)


sample_reviews = [

    "I absolutely love this dress. It fits perfectly and the quality is excellent.",

    "The product is terrible and the material is very poor.",

    "This clothing item is beautiful and comfortable.",

    "I am disappointed with this purchase. The quality is bad."

]


for review in sample_reviews:

    sentiment, probability = (
        predict_sentiment(
            review
        )
    )


    print("\nReview:")
    print(review)

    print(
        "Sentiment:",
        sentiment
    )

    print(
        "Confidence:",
        round(
            probability * 100,
            2
        ),
        "%"
    )


# ============================================================
# 27. REAL-TIME USER INPUT
# ============================================================

print("\n" + "=" * 60)
print("REAL-TIME E-COMMERCE REVIEW SENTIMENT ANALYZER")
print("=" * 60)


while True:

    user_review = input(
        "\nEnter a customer review "
        "(type 'exit' to stop): "
    )


    if user_review.lower() == "exit":

        break


    sentiment, probability = (
        predict_sentiment(
            user_review
        )
    )


    print(
        "\nPredicted Sentiment:",
        sentiment
    )


    print(
        "Confidence:",
        round(
            probability * 100,
            2
        ),
        "%"
    )


# ============================================================
# 28. SAVE MODEL
# ============================================================

torch.save(
    model.state_dict(),
    "lstm_sentiment_model.pth"
)


# Save vocabulary

import pickle

with open(
    "vocabulary.pkl",
    "wb"
) as file:

    pickle.dump(
        word_to_index,
        file
    )


print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print(
    "lstm_sentiment_model.pth"
)

print(
    "vocabulary.pkl"
)

print("\nPROGRAM COMPLETED")
print("=" * 60)

Device: cpu
LSTM CUSTOMER REVIEW SENTIMENT ANALYZER

Dataset Shape:
(23486, 11)

Columns:
['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']

First 5 Rows:
   Unnamed: 0  Clothing ID  Age                    Title  \
0           0          767   33                      NaN   
1           1         1080   34                      NaN   
2           2         1077   60  Some major design flaws   
3           3         1049   50         My favorite buy!   
4           4          847   47         Flattering shirt   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4                1   
1  Love this dress!  it's sooo pretty.  i happene...       5                1   
2  I had such high hopes for this dress and reall...       3                0   
3  I love, love, love this jumpsuit. it's f